# EDA анализ

## Import библиотек и загрузка данных

In [54]:
import pandas as pd

SEED = 42

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")
events = pd.read_csv("../data/events.csv.gz")

train.head()

,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


In [55]:
events.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
0,ck_5efbea1befdefe1b,2026-04-26 09:11:24,200,item_view,desktop,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1027450.0,elektronika,kaliningrad,pro,NaN,NaN,NaN,NaN
1,ck_c4ca1434f3778f1d,2026-04-20 14:04:31,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,telefony,habarovsk,NaN,iphone 13 128,4.0,NaN,NaN
2,ck_d274382b19488771,2026-04-13 12:02:56,200,item_view,WEB,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1048743.0,kvartiry_prodazha,kirov,private,NaN,NaN,675.0,276.0
3,ck_fbbed2ff14944ce9,2026-04-08 06:12:14,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,bytovaya_tehnika,novosibirsk,NaN,пылесос dyson,2.0,NaN,NaN
4,ck_56cc15c7c634cb9f,2026-04-12 17:16:05,100,search_results_view,WEB,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,odezhda,sankt-peterburg,NaN,костюм мужской,2.0,NaN,NaN


## Просмотр основной информации

In [56]:
for c in ["cookie_created_at", "window_start_ts", "window_end_ts"]:
    train[c] = pd.to_datetime(train[c])
    test[c]  = pd.to_datetime(test[c]) if c in test.columns else None
events["event_ts"] = pd.to_datetime(events["event_ts"])

date_cols = {
    "train": ["cookie_created_at", "window_start_ts", "window_end_ts"],
    "test": ["cookie_created_at", "window_start_ts", "window_end_ts"],
    "events": ["event_ts"],
}

for nm, df in [("train", train), ("test", test), ("events", events)]:
    print(nm, df.shape)
    print(df[date_cols[nm]].isna().sum())
    print()

train (11091, 5)
cookie_created_at    0
window_start_ts      0
window_end_ts        0
dtype: int64

test (4909, 4)
cookie_created_at    0
window_start_ts      0
window_end_ts        0
dtype: int64

events (328905, 14)
event_ts    0
dtype: int64



In [57]:
print(train.dtypes, "\n")
print(test.dtypes, "\n")
print(events.dtypes, "\n")

print("train shape:", train.shape, "| test shape:", test.shape, "| events shape:", events.shape)
print("память events, MB:", events.memory_usage(deep=True).sum() / 1e6)

print("\ncookie_id уникален в train:", train.cookie_id.is_unique)
print("cookie_id уникален в test:", test.cookie_id.is_unique)
print("пересечение cookie_id train/test:", len(set(train.cookie_id) & set(test.cookie_id)))

cookie_id                       str
cookie_created_at    datetime64[us]
window_start_ts      datetime64[us]
window_end_ts        datetime64[us]
target                        int64
dtype: object 

cookie_id                       str
cookie_created_at    datetime64[us]
window_start_ts      datetime64[us]
window_end_ts        datetime64[us]
dtype: object 

cookie_id                   str
event_ts         datetime64[us]
eid                       int64
event_name                  str
platform                    str
user_agent                  str
item_id                 float64
item_category               str
item_location               str
seller_type                 str
search_query                str
search_page             float64
pointer_x               float64
pointer_y               float64
dtype: object 

train shape: (11091, 5) | test shape: (4909, 4) | events shape: (328905, 14)
память events, MB: 90.817254

cookie_id уникален в train: True
cookie_id уникален в test: True
пересече

In [58]:
def case_audit(df, name):
    print(f"--- {name} ---")
    found = False
    for col in df.columns:
        if not pd.api.types.is_string_dtype(df[col]):
            continue
        vals = df[col].dropna()
        if vals.empty:
            continue
        n_raw = vals.nunique()
        n_lower = vals.str.lower().nunique()
        if n_raw != n_lower:
            print(f"{col}: {n_raw} значений -> {n_lower} после lower(), разница {n_raw - n_lower}")
            found = True
    if not found:
        print("расхождений не найдено")
    print()

case_audit(train, "train")
case_audit(test, "test")
case_audit(events, "events")

--- train ---
расхождений не найдено

--- test ---
расхождений не найдено

--- events ---
platform: 11 значений -> 5 после lower(), разница 6



In [59]:
print("до нормализации:", events.platform.nunique(), "вариантов")
print(events.platform.value_counts())

events["platform"] = events.platform.str.lower().replace({"iphone": "ios"})

print("\nпосле нормализации:", events.platform.nunique(), "варианта")
print(events.platform.value_counts())

до нормализации: 11 вариантов
platform
desktop    46866
WEB        46476
Web        46365
web        45951
ANDROID    42462
Android    42254
android    42159
iphone      4103
IOS         4100
iOS         4091
ios         4078
Name: count, dtype: int64

после нормализации: 4 варианта
platform
web        138792
android    126875
desktop     46866
ios         16372
Name: count, dtype: int64


In [60]:
def events_in_window(events, meta):
    ev = events.merge(meta[["cookie_id", "window_start_ts", "window_end_ts"]], on="cookie_id", how="inner")
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)].copy()

ev_win_tr = events_in_window(events, train)
ev_win_te = events_in_window(events, test)

print("train:", train.shape[0], "| test:", test.shape[0])
print("событий в окне (train):", len(ev_win_tr), "из", len(events[events.cookie_id.isin(train.cookie_id)]))
print("событий в окне (test):", len(ev_win_te), "из", len(events[events.cookie_id.isin(test.cookie_id)]))

print("\nбаланс классов в train:")
print(train.target.value_counts())
print(train.target.value_counts(normalize=True))

print("\nсобытий на train-куку в среднем (в окне):", len(ev_win_tr) / train.shape[0])
print("событий на test-куку в среднем (в окне):", len(ev_win_te) / test.shape[0])

print("\nкуки без событий в окне, train:", (~train.cookie_id.isin(ev_win_tr.cookie_id)).sum())
print("куки без событий в окне, test:", (~test.cookie_id.isin(ev_win_te.cookie_id)).sum())

train: 11091 | test: 4909
событий в окне (train): 198436 из 239215
событий в окне (test): 89690 из 89690

баланс классов в train:
target
0    10192
1      899
Name: count, dtype: int64
target
0    0.918943
1    0.081057
Name: proportion, dtype: float64

событий на train-куку в среднем (в окне): 17.89162383914886
событий на test-куку в среднем (в окне): 18.270523528213484

куки без событий в окне, train: 0
куки без событий в окне, test: 0


In [61]:
print("длительность окна train:", (train.window_end_ts - train.window_start_ts).value_counts())
print("длительность окна test:", (test.window_end_ts - test.window_start_ts).value_counts())

print("\nдиапазон train:", train.window_start_ts.min(), "—", train.window_start_ts.max())
print("диапазон test:", test.window_start_ts.min(), "—", test.window_start_ts.max())

print("\nокна по дням, train:")
print(train.window_start_ts.value_counts().sort_index())
print("\nокна по дням, test:")
print(test.window_start_ts.value_counts().sort_index())

print("\nдоля ботов по дням, train:")
print(train.groupby("window_start_ts").target.mean())

длительность окна train: 1 days    11091
Name: count, dtype: int64
длительность окна test: 1 days    4909
Name: count, dtype: int64

диапазон train: 2026-04-06 00:00:00 — 2026-04-19 00:00:00
диапазон test: 2026-04-20 00:00:00 — 2026-04-26 00:00:00

окна по дням, train:
window_start_ts
2026-04-06    772
2026-04-07    797
2026-04-08    834
2026-04-09    902
2026-04-10    912
2026-04-11    896
2026-04-12    817
2026-04-13    843
2026-04-14    826
2026-04-15    801
2026-04-16    740
2026-04-17    713
2026-04-18    606
2026-04-19    632
Name: count, dtype: int64

окна по дням, test:
window_start_ts
2026-04-20    652
2026-04-21    628
2026-04-22    637
2026-04-23    660
2026-04-24    760
2026-04-25    759
2026-04-26    813
Name: count, dtype: int64

доля ботов по дням, train:
window_start_ts
2026-04-06    0.088083
2026-04-07    0.074028
2026-04-08    0.083933
2026-04-09    0.073171
2026-04-10    0.089912
2026-04-11    0.088170
2026-04-12    0.082007
2026-04-13    0.062871
2026-04-14    0.084

In [62]:
train["cookie_age_days"] = (train.window_start_ts - train.cookie_created_at).dt.total_seconds() / 86400
test["cookie_age_days"]  = (test.window_start_ts - test.cookie_created_at).dt.total_seconds() / 86400

print("отрицательный возраст (создана после начала окна), train:", (train.cookie_age_days < 0).sum())
print("отрицательный возраст, test:", (test.cookie_age_days < 0).sum())

print("\nвозраст куки по классам, train:")
print(train.groupby("target").cookie_age_days.describe())

print("\nвозраст куки, test (весь):")
print(test.cookie_age_days.describe())

отрицательный возраст (создана после начала окна), train: 0
отрицательный возраст, test: 0

возраст куки по классам, train:
          count       mean        std       min        25%        50%  \
target                                                                  
0       10192.0  73.908394  66.970965  0.014502  18.850226  61.782367   
1         899.0  48.726411  62.795697  0.012361   0.943495  18.968403   

              75%         max  
target                         
0       110.44640  449.284873  
1        83.45294  361.764005  

возраст куки, test (весь):
count    4909.000000
mean       71.647038
std        67.999234
min         0.020324
25%        12.138646
50%        58.631076
75%       107.642708
max       465.293692
Name: cookie_age_days, dtype: float64


In [63]:
m = pd.concat([train[["cookie_id", "window_start_ts", "window_end_ts", "target"]],
               test.assign(target=pd.NA)[["cookie_id", "window_start_ts", "window_end_ts", "target"]]])

e = events.merge(m, on="cookie_id", how="inner")
e["before"] = e.event_ts < e.window_start_ts
e["after"]  = e.event_ts >= e.window_end_ts

print("событий до окна:", e.before.sum(), "| событий после окна:", e.after.sum(), "| всего:", len(e))

print("\nтипы событий после окна (все куки):")
print(e[e.after].event_name.value_counts())

print("\nтипы событий после окна, только captcha_shown — есть ли она вообще ДО или ВНУТРИ окна:")
capt = e[e.event_name == "captcha_shown"]
print("captcha всего:", len(capt), "| до окна:", capt.before.sum(), "| внутри окна:", (~capt.before & ~capt.after).sum(), "| после окна:", capt.after.sum())

tr_only = e[e.target.notna()].copy()
tr_only["target"] = tr_only.target.astype(int)
g = tr_only.groupby("cookie_id").agg(target=("target", "first"), n_after=("after", "sum"))
print("\nсреднее число событий после окна по классам (train):")
print(g.groupby("target").n_after.agg(["mean", "median", lambda s: (s > 0).mean()]))

событий до окна: 0 | событий после окна: 40779 | всего: 328905

типы событий после окна (все куки):
event_name
item_view               12731
search_results_view     10604
captcha_shown            7928
photo_swipe              3433
seller_page_view         1861
favorite_add             1753
contact_phone_show       1028
login                     606
contact_chat_open         577
contact_message_sent      258
Name: count, dtype: int64

типы событий после окна, только captcha_shown — есть ли она вообще ДО или ВНУТРИ окна:
captcha всего: 7928 | до окна: 0 | внутри окна: 0 | после окна: 7928

среднее число событий после окна по классам (train):
             mean  median  <lambda_0>
target                               
0        2.080651     0.0    0.450157
1       21.771969    26.0    0.769744


In [64]:
# полные дубликаты
full_dup = events.duplicated()
print("полных дублей в events:", full_dup.sum(), f"({full_dup.mean():.3%})")

# дубли по смысловому ключу
key_dup = events.duplicated(subset=["cookie_id", "event_ts", "eid"])
print("дублей по (cookie_id, event_ts, eid):", key_dup.sum())

# концентрируются ли дубли у небольшого числа кук
dup_by_cookie = events[full_dup].cookie_id.value_counts()
print("\nкук с дублями:", dup_by_cookie.shape[0], "| максимум дублей у одной куки:", dup_by_cookie.max())

# дубли по классам (используем train-часть)
d = events.merge(train[["cookie_id", "target"]], on="cookie_id", how="inner")
d["dup"] = d.duplicated(subset=events.columns)
print("\nдоля дублей по классам (train):")
print(d.groupby("target").dup.mean())

# пропуски по колонкам
print("\nдоля пропусков по колонкам (весь events):")
print(events.isna().mean().round(3).sort_values(ascending=False))

# пропуски в разрезе event_name
print("\nдоля пропусков item_id / search_page / pointer_x по event_name:")
print(events.groupby("event_name")[["item_id", "search_page", "pointer_x"]].apply(lambda g: g.isna().mean()))

полных дублей в events: 4868 (1.480%)
дублей по (cookie_id, event_ts, eid): 5270

кук с дублями: 3777 | максимум дублей у одной куки: 6

доля дублей по классам (train):
target
0    0.014855
1    0.014258
Name: dup, dtype: float64

доля пропусков по колонкам (весь events):
search_query     0.695
search_page      0.695
pointer_x        0.670
pointer_y        0.670
seller_type      0.407
item_id          0.348
item_category    0.100
item_location    0.072
user_agent       0.000
platform         0.000
event_ts         0.000
cookie_id        0.000
event_name       0.000
eid              0.000
dtype: float64

доля пропусков item_id / search_page / pointer_x по event_name:
                      item_id  search_page  pointer_x
event_name                                           
captcha_shown             1.0          1.0   0.727800
contact_chat_open         0.0          1.0   0.691755
contact_message_sent      0.0          1.0   0.677377
contact_phone_show        0.0          1.0   0.676741
f

In [65]:
key_cols = ["cookie_id", "event_ts", "eid"]
dup_mask = events.duplicated(subset=key_cols, keep=False)
same_key = events[dup_mask].sort_values(key_cols)

# среди строк с одинаковым ключом — сколько НЕ идентичны полностью
n_groups = same_key.groupby(key_cols).ngroups
n_full_dup_in_groups = same_key.duplicated().sum()
print("групп с одинаковым (cookie_id, event_ts, eid):", n_groups)
print("строк в этих группах:", len(same_key))
print("из них полных дублей:", n_full_dup_in_groups, "| различающихся:", len(same_key) - n_full_dup_in_groups - n_groups)

# пример различающихся строк
diff_rows = same_key[~same_key.duplicated(keep=False) | ~same_key.duplicated()]
print(same_key.groupby(key_cols).filter(lambda g: g.drop_duplicates().shape[0] > 1).head(10))

групп с одинаковым (cookie_id, event_ts, eid): 5260
строк в этих группах: 10530
из них полных дублей: 4868 | различающихся: 402
                  cookie_id            event_ts  eid           event_name  \
5554    ck_0010ec3874fb5378 2026-04-06 17:56:50  100  search_results_view   
105587  ck_0010ec3874fb5378 2026-04-06 17:56:50  100  search_results_view   
299065  ck_00fc4e4bf18b0742 2026-04-13 20:46:07  100  search_results_view   
310520  ck_00fc4e4bf18b0742 2026-04-13 20:46:07  100  search_results_view   
116662  ck_01563097de731a96 2026-04-17 20:11:56  200            item_view   
274871  ck_01563097de731a96 2026-04-17 20:11:56  200            item_view   
113022  ck_01e05a9ddde86327 2026-04-12 17:17:48  200            item_view   
303489  ck_01e05a9ddde86327 2026-04-12 17:17:48  200            item_view   
297803  ck_024dcab126876002 2026-04-23 17:53:40  200            item_view   
326543  ck_024dcab126876002 2026-04-23 17:53:40  200            item_view   

       platform         

In [66]:
n_before = len(events)
events = events.drop_duplicates().reset_index(drop=True)
print("удалено дублей:", n_before - len(events), "| осталось строк:", len(events))

удалено дублей: 4868 | осталось строк: 324037


In [67]:
print("доля пропусков pointer_x по platform:")
print(events.groupby("platform").pointer_x.apply(lambda s: s.isna().mean()))

print("\nдоля пропусков pointer_x по platform и event_name (только web/desktop):")
sub = events[events.platform.isin(["web", "desktop"])]
print(sub.groupby(["platform", "event_name"]).pointer_x.apply(lambda s: s.isna().mean()).round(3))

доля пропусков pointer_x по platform:
platform
android    1.000000
desktop    0.413242
ios        1.000000
web        0.416000
Name: pointer_x, dtype: float64

доля пропусков pointer_x по platform и event_name (только web/desktop):
platform  event_name          
desktop   captcha_shown           0.562
          contact_chat_open       0.383
          contact_message_sent    0.319
          contact_phone_show      0.356
          favorite_add            0.345
          item_view               0.423
          login                   0.343
          photo_swipe             0.372
          search_results_view     0.431
          seller_page_view        0.400
web       captcha_shown           0.586
          contact_chat_open       0.363
          contact_message_sent    0.364
          contact_phone_show      0.358
          favorite_add            0.349
          item_view               0.435
          login                   0.347
          photo_swipe             0.364
          search_

In [68]:
# 1. соответствие eid <-> event_name
print("eid <-> event_name:")
print(events.groupby(["eid", "event_name"]).size())
print("\nуникальных eid:", events.eid.nunique(), "| уникальных event_name:", events.event_name.nunique())

# 2. кардинальность категориальных полей
for col in ["item_category", "item_location", "seller_type", "search_page"]:
    print(f"\n{col}: {events[col].nunique()} уникальных значений")
    print(events[col].value_counts(dropna=False).head(10))

# 3. согласованность platform и user_agent
def ua_platform_guess(ua):
    ua = ua.lower()
    if "android" in ua:
        return "android"
    if "iphone" in ua or "ios" in ua:
        return "ios"
    if any(x in ua for x in ["windows", "macintosh", "linux", "x11"]):
        return "web_or_desktop"
    return "unknown"

events["ua_guess"] = events.user_agent.apply(ua_platform_guess)
print("\nplatform vs UA-угадка:")
print(pd.crosstab(events.platform, events.ua_guess))

# 4. headless-маркеры в UA
events["is_headless"] = events.user_agent.str.contains("Headless", case=False, na=False)
print("\ndoля headless по platform:")
print(events.groupby("platform").is_headless.mean())

eid <-> event_name:
eid  event_name          
100  search_results_view      98928
200  item_view               118989
210  photo_swipe              35962
220  seller_page_view         17180
300  contact_phone_show       11154
301  contact_chat_open         6047
303  contact_message_sent      3032
400  favorite_add             18739
500  login                     6177
900  captcha_shown             7829
dtype: int64

уникальных eid: 10 | уникальных event_name: 10

item_category: 15 уникальных значений
item_category
NaN                  32455
avtomobili           20536
mebel                20236
bytovaya_tehnika     19861
hobbi                19731
elektronika          19711
uslugi               19586
odezhda              19559
kvartiry_arenda      19428
kvartiry_prodazha    19404
Name: count, dtype: int64

item_location: 40 уникальных значений
item_location
sankt-peterburg    38579
novosibirsk        37447
moskva             37385
ekaterinburg       36732
NaN                23465
ryazan

In [69]:
# 1. что за UA не распознались как windows/mac/linux
unknown_ua = events[(events.platform.isin(["desktop", "web"])) & (events.ua_guess == "unknown")]
print("уникальных UA в unknown:", unknown_ua.user_agent.nunique())
print(unknown_ua.user_agent.value_counts().head(15))

# 2. headless по классам (уже можно смотреть, раз мы в train/test это умеем мёржить)
e = events.merge(
    pd.concat([train[["cookie_id","target"]], test.assign(target=pd.NA)[["cookie_id","target"]]]),
    on="cookie_id", how="inner"
)
tr_e = e[e.target.notna()].copy()
tr_e["target"] = tr_e.target.astype(int)
print("\nдоля headless-событий по target (train):")
print(tr_e.groupby("target").is_headless.mean())

g = tr_e.groupby("cookie_id").agg(target=("target","first"), any_headless=("is_headless","max"))
print("\nдоля кук хоть раз headless, по классам:")
print(g.groupby("target").any_headless.mean())

# 3. полный список категорий и топ городов
print("\nвсе item_category:")
print(events.item_category.value_counts(dropna=False))
print("\nхвост item_location (редкие города):")
print(events.item_location.value_counts(dropna=False).tail(15))

уникальных UA в unknown: 6
user_agent
node-fetch/3.3.2                       1275
curl/8.1.2                             1195
Scrapy/2.11.0 (+https://scrapy.org)    1190
python-urllib3/2.0.7                   1180
Go-http-client/1.1                     1179
python-requests/2.31.0                  961
Name: count, dtype: int64

доля headless-событий по target (train):
target
0    0.020145
1    0.111397
Name: is_headless, dtype: float64

доля кук хоть раз headless, по классам:
target
0    0.020703
1    0.083426
Name: any_headless, dtype: float64

все item_category:
item_category
NaN                  32455
avtomobili           20536
mebel                20236
bytovaya_tehnika     19861
hobbi                19731
elektronika          19711
uslugi               19586
odezhda              19559
kvartiry_arenda      19428
kvartiry_prodazha    19404
noutbuki             19277
detskie_tovary       19139
telefony             19099
rabota               19083
zhivotnye            18823
zapchasti  

In [70]:
print("пропуски item_category по event_name:")
print(events.groupby("event_name").item_category.apply(lambda s: s.isna().mean()))

пропуски item_category по event_name:
event_name
captcha_shown           1.000000
contact_chat_open       0.058211
contact_message_sent    0.060356
contact_phone_show      0.058903
favorite_add            0.063290
item_view               0.059400
login                   1.000000
photo_swipe             0.058312
search_results_view     0.059649
seller_page_view        0.058498
Name: item_category, dtype: float64


In [71]:
raw_clients = ["node-fetch", "curl/", "Scrapy", "python-urllib3", "Go-http-client", "python-requests"]
events["is_raw_client"] = events.user_agent.str.contains("|".join(raw_clients), case=False, na=False, regex=True)

tr_e = e[e.target.notna()].copy()
tr_e["target"] = tr_e.target.astype(int)
tr_e["is_raw_client"] = events.loc[tr_e.index, "is_raw_client"] if False else tr_e.cookie_id.map(events.groupby("cookie_id").is_raw_client.max())

print("доля событий is_raw_client по target:")
print(events.merge(train[["cookie_id","target"]], on="cookie_id").groupby("target").is_raw_client.mean())

g = events.merge(train[["cookie_id","target"]], on="cookie_id").groupby("cookie_id").agg(
    target=("target","first"), any_raw_client=("is_raw_client","max"), any_headless=("is_headless","max")
)
print("\nдоля кук хоть раз raw_client, по классам:")
print(g.groupby("target").any_raw_client.mean())

print("\nобъединённый флаг (raw_client ИЛИ headless):")
g["any_bot_ua"] = g.any_raw_client | g.any_headless
print(g.groupby("target").any_bot_ua.mean())

доля событий is_raw_client по target:
target
0    0.014343
1    0.056667
Name: is_raw_client, dtype: float64

доля кук хоть раз raw_client, по классам:
target
0    0.015306
1    0.048943
Name: any_raw_client, dtype: float64

объединённый флаг (raw_client ИЛИ headless):
target
0    0.036009
1    0.132369
Name: any_bot_ua, dtype: float64


In [72]:
def cookie_profile(ev, meta):
    ev = ev.sort_values(["cookie_id", "event_ts"])
    g = ev.groupby("cookie_id")

    diffs = ev.groupby("cookie_id").event_ts.diff().dt.total_seconds()
    ev = ev.assign(gap=diffs)

    prof = g.agg(
        n_events=("event_ts", "size"),
        n_event_types=("event_name", "nunique"),
        n_platforms=("platform", "nunique"),
        span_sec=("event_ts", lambda s: (s.max() - s.min()).total_seconds()),
    )
    gap_stats = ev.groupby("cookie_id").gap.agg(
        gap_mean="mean", gap_median="median", gap_std="std", gap_min="min"
    )
    prof = prof.join(gap_stats)
    prof["n_sessions"] = ev.assign(new_session=(ev.gap.isna()) | (ev.gap > 1800)).groupby("cookie_id").new_session.sum()

    prof = meta[["cookie_id"]].merge(prof.reset_index(), on="cookie_id", how="left")
    return prof

# используем только события внутри окна (ev_win_tr/ev_win_te уже без дублей, если пересобрали после drop_duplicates)
prof_tr = cookie_profile(ev_win_tr, train)
prof_tr = prof_tr.merge(train[["cookie_id", "target"]], on="cookie_id")

print(prof_tr.groupby("target")[["n_events", "n_event_types", "n_platforms", "span_sec",
                                   "gap_mean", "gap_median", "gap_std", "gap_min", "n_sessions"]].median())

        n_events  n_event_types  n_platforms  span_sec    gap_mean  \
target                                                               
0           12.0            4.0          1.0    8326.5  657.294118   
1           20.0            5.0          2.0    9092.0  425.407407   

        gap_median      gap_std  gap_min  n_sessions  
target                                                
0             59.0  1732.919959      5.0         2.0  
1             25.0  1368.461303      3.0         2.0  


In [73]:
print("распределение n_platforms по классам:")
print(prof_tr.groupby("target").n_platforms.value_counts(normalize=True).unstack())

# смотрим на примеры кук с >1 платформой
multi = ev_win_tr[ev_win_tr.cookie_id.isin(
    ev_win_tr.groupby("cookie_id").platform.nunique().loc[lambda s: s > 1].index
)]
print("\nпримеры смены платформы внутри одной куки:")
print(multi.sort_values(["cookie_id","event_ts"])[["cookie_id","event_ts","platform","event_name"]].head(20))

# полная сводка по всем метрикам, не только медиана
print("\nполная статистика по target:")
print(prof_tr.groupby("target")[["n_events","n_event_types","n_platforms","span_sec",
                                   "gap_mean","gap_median","gap_std","gap_min","n_sessions"]].describe().T)

распределение n_platforms по классам:
n_platforms         1         2
target                         
0            0.543564  0.456436
1            0.369299  0.630701

примеры смены платформы внутри одной куки:
                  cookie_id            event_ts platform           event_name
189443  ck_000c95f1408dcb00 2026-04-19 14:49:54      web  search_results_view
47919   ck_000c95f1408dcb00 2026-04-19 14:50:17      web          photo_swipe
143776  ck_000c95f1408dcb00 2026-04-19 15:50:21      web            item_view
134951  ck_000c95f1408dcb00 2026-04-19 15:50:47      web            item_view
155957  ck_000c95f1408dcb00 2026-04-19 15:51:45      web            item_view
49261   ck_000c95f1408dcb00 2026-04-19 16:56:24      web            item_view
152324  ck_000c95f1408dcb00 2026-04-19 16:56:44      web            item_view
164076  ck_000c95f1408dcb00 2026-04-19 19:05:37      web            item_view
216231  ck_000c95f1408dcb00 2026-04-19 19:06:45      web  search_results_view
4098    ck

In [74]:
# для кук с 2 платформами - какая доля событий приходится на "младшую" (реже встречающуюся) платформу
multi_cookies = ev_win_tr.groupby("cookie_id").platform.nunique()
multi_cookies = multi_cookies[multi_cookies > 1].index

def minority_share(g):
    counts = g.platform.value_counts()
    return counts.min() / counts.sum()

minority = ev_win_tr[ev_win_tr.cookie_id.isin(multi_cookies)].groupby("cookie_id").apply(minority_share, include_groups=False)
minority = minority.rename("minority_share").reset_index().merge(train[["cookie_id","target"]], on="cookie_id")

print("доля событий на редкой платформе, по классам (среди кук с 2 платформами):")
print(minority.groupby("target").minority_share.describe())

# какие именно пары платформ встречаются чаще всего
pairs = ev_win_tr[ev_win_tr.cookie_id.isin(multi_cookies)].groupby("cookie_id").platform.apply(lambda s: tuple(sorted(s.unique())))
pairs = pairs.rename("platform_pair").reset_index().merge(train[["cookie_id","target"]], on="cookie_id")
print("\nсамые частые пары платформ, по классам:")
print(pairs.groupby("target").platform_pair.value_counts(normalize=True).groupby(level=0).head(5))

доля событий на редкой платформе, по классам (среди кук с 2 платформами):
         count      mean       std       min  25%       50%       75%  max
target                                                                    
0       4652.0  0.269738  0.104610  0.040000  0.2  0.258065  0.333333  0.5
1        567.0  0.257702  0.091622  0.045455  0.2  0.250000  0.305235  0.5

самые частые пары платформ, по классам:
target  platform_pair 
0       (desktop, web)    1.0
1       (desktop, web)    1.0
Name: proportion, dtype: float64


In [75]:
bins = [0, 5, 10, 20, 40, 1000]
prof_tr["n_events_bin"] = pd.cut(prof_tr.n_events, bins)
print(prof_tr.groupby(["n_events_bin", "target"]).apply(lambda g: (g.n_platforms > 1).mean(), include_groups=False))

n_events_bin  target
(0, 5]        0         0.284753
              1         0.419753
(5, 10]       0         0.468094
              1         0.608696
(10, 20]      0         0.520302
              1         0.524793
(20, 40]      0         0.516940
              1         0.665217
(40, 1000]    0         0.526502
              1         0.812500
dtype: float64


In [ ]:
sub = ev_win_tr[ev_win_tr.cookie_id.isin(multi_cookies)]

def ua_by_platform(g):
    d = g.groupby("platform").user_agent.apply(lambda s: set(s))
    return d.get("desktop", set()) == d.get("web", set())

same_ua = sub.groupby("cookie_id").apply(ua_by_platform, include_groups=False)
print("доля кук, где UA одинаковый под desktop и web:", same_ua.mean())

# несколько примеров вживую
for cid in same_ua.index[:5]:
    d = sub[sub.cookie_id == cid].groupby("platform").user_agent.unique()
    print(cid, dict(d))

доля кук, где UA одинаковый под desktop и web: 0.9168423069553554
ck_000c95f1408dcb00 {'desktop': <ArrowStringArray>
['Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36']
Length: 1, dtype: str, 'web': <ArrowStringArray>
['Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36']
Length: 1, dtype: str}
ck_000e8c52636e3bec {'desktop': <ArrowStringArray>
['Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:116.0) Gecko/20100101 Firefox/116.0']
Length: 1, dtype: str, 'web': <ArrowStringArray>
['Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:116.0) Gecko/20100101 Firefox/116.0']
Length: 1, dtype: str}
ck_0010e31baa4a1fb7 {'desktop': <ArrowStringArray>
['Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/112.0.0.0 YaBrowser/23.4.0.0 Safari/537.36']
Length: 1, dtype: str, 'web': <ArrowStringArray>
['Mozilla/5.0 (Windows NT 10.0; Win64; 

In [80]:
def browser_family(ua):
    ua = ua.lower()
    if "headlesschrome" in ua: return "headless_chrome"
    if "yabrowser" in ua: return "yandex"
    if "edg" in ua: return "edge"
    if "opr" in ua or "opera" in ua: return "opera"
    if "firefox" in ua: return "firefox"
    if "chrome" in ua: return "chrome"
    if "safari" in ua: return "safari"
    return "other"

ev_win_tr["browser_family"] = ev_win_tr.user_agent.apply(browser_family)

ua_div = ev_win_tr.groupby("cookie_id").agg(
    n_ua=("user_agent", "nunique"),
    n_family=("browser_family", "nunique"),
).reset_index().merge(train[["cookie_id", "target"]], on="cookie_id")

print("доля кук с >1 уникальным user_agent (включая версии), по классам:")
print(ua_div.groupby("target").apply(lambda g: (g.n_ua > 1).mean(), include_groups=False))

print("\nдоля кук с >1 РАЗНЫМ браузерным семейством (не версия, а именно другой браузер), по классам:")
print(ua_div.groupby("target").apply(lambda g: (g.n_family > 1).mean(), include_groups=False))

# что происходит в тех редких случаях, где семейство меняется
diff_family = ev_win_tr[ev_win_tr.cookie_id.isin(
    ua_div[ua_div.n_family > 1].cookie_id
)]
print("\nпримеры кук с реально разными семействами браузера:")
print(diff_family.sort_values(["cookie_id","event_ts"])[["cookie_id","event_ts","platform","browser_family","user_agent"]].head(20))

доля кук с >1 уникальным user_agent (включая версии), по классам:
target
0    0.073097
1    0.056730
dtype: float64

доля кук с >1 РАЗНЫМ браузерным семейством (не версия, а именно другой браузер), по классам:
target
0    0.059557
1    0.047831
dtype: float64

примеры кук с реально разными семействами браузера:
                  cookie_id            event_ts platform browser_family  \
136026  ck_008c04d82d0561f5 2026-04-06 04:15:03      web         chrome   
16763   ck_008c04d82d0561f5 2026-04-06 04:15:15  desktop         yandex   
129941  ck_008c04d82d0561f5 2026-04-06 04:15:24  desktop         chrome   
53508   ck_008c04d82d0561f5 2026-04-06 04:15:41      web        firefox   
4450    ck_008c04d82d0561f5 2026-04-06 04:16:30  desktop        firefox   
68146   ck_008c04d82d0561f5 2026-04-06 04:17:29      web         yandex   
216503  ck_008c04d82d0561f5 2026-04-06 04:17:56      web        firefox   
133579  ck_008c04d82d0561f5 2026-04-06 04:18:52      web        firefox   
108451  ck_0

In [81]:
print(train[train.cookie_id.isin(["ck_008c04d82d0561f5", "ck_008ed1631c1bf43b"])])

                cookie_id   cookie_created_at window_start_ts window_end_ts  \
360   ck_008c04d82d0561f5 2026-01-02 23:57:46      2026-04-06    2026-04-07   
3124  ck_008ed1631c1bf43b 2026-04-08 14:30:31      2026-04-09    2026-04-10   

      target  cookie_age_days  
360        0        93.001551  
3124       0         0.395475  


In [78]:
events["platform_canon"] = events.platform.replace({"desktop": "web"})
print(events.platform_canon.value_counts())

platform_canon
web        182943
android    124984
ios         16110
Name: count, dtype: int64


In [79]:
prof_tr["n_platforms_canon"] = ev_win_tr.assign(
    platform_canon=ev_win_tr.platform.replace({"desktop": "web"})
).groupby("cookie_id").platform_canon.nunique().reindex(prof_tr.cookie_id).values

print("доля кук с настоящей мультиплатформенностью (после объединения desktop+web), по классам:")
print(prof_tr.merge(train[["cookie_id","target"]], on="cookie_id", suffixes=("","_y")) \
      .groupby("target").n_platforms_canon.apply(lambda s: (s > 1).mean()))

доля кук с настоящей мультиплатформенностью (после объединения desktop+web), по классам:
target
0    0.0
1    0.0
Name: n_platforms_canon, dtype: float64


In [82]:
# распределение по часам суток
ev_win_tr["hour"] = ev_win_tr.event_ts.dt.hour
hourly = ev_win_tr.merge(train[["cookie_id","target"]], on="cookie_id")
print("доля событий по часам, норм. внутри класса:")
print(pd.crosstab(hourly.hour, hourly.target, normalize="columns").round(3))

night = hourly.hour.between(2, 5)
print("\nдоля событий в 2:00-5:59 по классам:")
print(hourly.assign(night=night).groupby("target").night.mean())

# биграммы последовательных событий внутри куки
ev_sorted = ev_win_tr.sort_values(["cookie_id", "event_ts"])
ev_sorted["next_event"] = ev_sorted.groupby("cookie_id").event_name.shift(-1)
bigrams = ev_sorted.dropna(subset=["next_event"]).merge(train[["cookie_id","target"]], on="cookie_id")
bigrams["pair"] = list(zip(bigrams.event_name, bigrams.next_event))

top_bigrams_bot = bigrams[bigrams.target==1].pair.value_counts(normalize=True).head(8)
top_bigrams_human = bigrams[bigrams.target==0].pair.value_counts(normalize=True).head(8)
print("\nтоп биграмм у ботов:")
print(top_bigrams_bot)
print("\nтоп биграмм у людей:")
print(top_bigrams_human)

доля событий по часам, норм. внутри класса:
target      0      1
hour                
0       0.010  0.025
1       0.014  0.031
2       0.021  0.043
3       0.025  0.056
4       0.028  0.040
5       0.032  0.048
6       0.029  0.036
7       0.032  0.040
8       0.033  0.046
9       0.036  0.040
10      0.040  0.036
11      0.044  0.040
12      0.050  0.033
13      0.048  0.048
14      0.051  0.045
15      0.052  0.038
16      0.050  0.034
17      0.052  0.041
18      0.055  0.042
19      0.062  0.041
20      0.059  0.059
21      0.060  0.045
22      0.062  0.045
23      0.055  0.048

доля событий в 2:00-5:59 по классам:
target
0    0.106096
1    0.186177
Name: night, dtype: float64

топ биграмм у ботов:
pair
(item_view, item_view)                        0.197516
(search_results_view, item_view)              0.152335
(item_view, search_results_view)              0.152296
(search_results_view, search_results_view)    0.132381
(item_view, photo_swipe)                      0.030342
(photo_

In [83]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

def build_adv_features(ev, meta):
    ev = ev.copy()
    ev["platform_canon"] = ev.platform.replace({"desktop": "web"})
    g = ev.groupby("cookie_id")
    f = pd.DataFrame({
        "n_events": g.size(),
        "n_event_types": g.event_name.nunique(),
        "item_nunique": g.item_id.nunique(),
        "category_nunique": g.item_category.nunique(),
        "is_headless": g.apply(lambda x: x.user_agent.str.contains("Headless", case=False, na=False).max(), include_groups=False),
    })
    f = meta[["cookie_id"]].merge(f.reset_index(), on="cookie_id", how="left").fillna(0)
    return f.set_index("cookie_id")

adv_tr = build_adv_features(ev_win_tr, train)
adv_te = build_adv_features(ev_win_te, test)

adv_tr["is_test"] = 0
adv_te["is_test"] = 1
adv = pd.concat([adv_tr, adv_te])
X = adv.drop(columns="is_test")
y = adv.is_test

clf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=SEED)
scores = cross_val_score(clf, X, y, cv=5, scoring="roc_auc")
print("adversarial AUC (train vs test):", scores.mean(), "+-", scores.std())

clf.fit(X, y)
print("\nважность признаков:")
print(pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False))

adversarial AUC (train vs test): 0.4967915689088979 +- 0.006103174995964016

важность признаков:
n_events            0.444504
item_nunique        0.293780
n_event_types       0.133922
category_nunique    0.117407
is_headless         0.010387
dtype: float64


In [84]:
sq = ev_win_tr[ev_win_tr.event_name == "search_results_view"].merge(train[["cookie_id","target"]], on="cookie_id")

print("доля пустых search_query среди search_results_view:", sq.search_query.isna().mean())

print("\nдлина запроса (в символах), по классам:")
print(sq.assign(qlen=sq.search_query.str.len()).groupby("target").qlen.describe())

print("\nчисло слов в запросе, по классам:")
print(sq.assign(nwords=sq.search_query.str.split().str.len()).groupby("target").nwords.describe())

# разнообразие запросов на куку
div = sq.groupby("cookie_id").agg(
    n_queries=("search_query", "count"),
    n_unique_queries=("search_query", "nunique"),
).reset_index().merge(train[["cookie_id","target"]], on="cookie_id")
div["query_diversity"] = div.n_unique_queries / div.n_queries.clip(lower=1)

print("\nразнообразие запросов на куку (доля уникальных среди всех), по классам:")
print(div.groupby("target").query_diversity.describe())

# повторяет ли кука один и тот же запрос подряд
sq_sorted = sq.sort_values(["cookie_id", "event_ts"])
sq_sorted["same_as_prev"] = sq_sorted.groupby("cookie_id").search_query.shift() == sq_sorted.search_query
print("\nдоля повторов запроса подряд, по классам:")
print(sq_sorted.groupby("target").same_as_prev.mean())

# топ самых частых запросов и не встречаются ли одинаковые "странные" запросы массово у ботов
print("\nсамые частые запросы у ботов:")
print(sq[sq.target==1].search_query.value_counts().head(10))
print("\nсамые частые запросы у людей:")
print(sq[sq.target==0].search_query.value_counts().head(10))

доля пустых search_query среди search_results_view: 0.0

длина запроса (в символах), по классам:
          count       mean       std  min   25%   50%   75%   max
target                                                           
0       52399.0  13.226607  4.175606  4.0  11.0  13.0  16.0  24.0
1        9318.0  13.384417  4.088442  4.0  11.0  14.0  16.0  24.0

число слов в запросе, по классам:
          count      mean       std  min  25%  50%  75%  max
target                                                      
0       52399.0  1.993111  0.672432  1.0  2.0  2.0  2.0  4.0
1        9318.0  1.978643  0.674021  1.0  2.0  2.0  2.0  4.0

разнообразие запросов на куку (доля уникальных среди всех), по классам:
         count      mean       std       min       25%       50%  75%  max
target                                                                    
0       9308.0  0.840759  0.190589  0.133333  0.692308  1.000000  1.0  1.0
1        862.0  0.652358  0.263922  0.125000  0.438125  0.6666

In [85]:
q_stats = sq.groupby("search_query").agg(
    n_total=("cookie_id", "size"),
    n_cookies=("cookie_id", "nunique"),
    bot_share=("target", "mean"),
).sort_values("n_total", ascending=False)

print("топ запросов по числу РАЗНЫХ кук, использовавших их, и доле ботов среди них:")
print(q_stats.head(20))

# насколько сконцентрированы запросы ботов по сравнению с людьми
print("\nсреднее число кук на один уникальный запрос, по классам:")
for t, g in sq.groupby("target"):
    per_query = g.groupby("search_query").cookie_id.nunique()
    print(f"target={t}: среднее={per_query.mean():.2f}, медиана={per_query.median():.1f}, макс={per_query.max()}")

топ запросов по числу РАЗНЫХ кук, использовавших их, и доле ботов среди них:
                       n_total  n_cookies  bot_share
search_query                                        
шкаф купе                  599        409   0.146912
джинсы levis               589        377   0.203735
toyota camry               580        388   0.143103
renault logan              576        385   0.145833
вторичка 2к                575        382   0.139130
работа водителем           568        364   0.191901
kia rio 2019               564        386   0.164894
skoda octavia              558        372   0.211470
коньки                     558        394   0.137993
самокат детский            557        375   0.132855
iphone 13 128              557        387   0.138241
смена 12 часов             556        369   0.197842
апартаменты купить         555        356   0.160360
volkswagen polo            554        378   0.180505
xiaomi redmi note 12       553        366   0.180832
новостройка           

In [86]:
n_search_cookies = sq.groupby("target").cookie_id.nunique()
print("кук, хоть раз искавших, по классам:", n_search_cookies.to_dict())

n_unique_queries = sq.groupby("target").search_query.nunique()
print("уникальных текстов запроса, по классам:", n_unique_queries.to_dict())

# доля кук СВОЕГО класса, разделяющих один и тот же запрос - нормировано на размер класса
for t, g in sq.groupby("target"):
    n_search = g.cookie_id.nunique()
    per_query = g.groupby("search_query").cookie_id.nunique()
    share = per_query / n_search
    print(f"target={t}: доля кук класса на один запрос — среднее {share.mean():.4f}, медиана {share.median():.4f}")

кук, хоть раз искавших, по классам: {0: 9308, 1: 862}
уникальных текстов запроса, по классам: {0: 120, 1: 120}
target=0: доля кук класса на один запрос — среднее 0.0346, медиана 0.0344
target=1: доля кук класса на один запрос — среднее 0.0414, медиана 0.0412
